In [14]:

import numpy as np
import itertools
from datetime import date, timedelta
from tqdm import tqdm
from utils import scenario_name
import glob
import re
import datetime
import os
from collections import defaultdict
import networkx as nx
import pickle
import pandas as pd
import torch
from torch_geometric.data import HeteroData, Data

FEATURE_DIR = "scenario_features_batch" 

# 1. List all CSV feature files
all_paths = glob.glob(os.path.join(FEATURE_DIR, "*.csv"))

# Paths
WEEKLY_DIR      = "weekly_features"
EDGE_MASTER     = "edges_master2.csv"
HETERO_OUT_DIR  = "hetero_graphs"
HOMO_OUT_DIR    = "projected_graphs"
os.makedirs(HETERO_OUT_DIR, exist_ok=True)
os.makedirs(HOMO_OUT_DIR, exist_ok=True)
os.makedirs(WEEKLY_DIR, exist_ok=True)

# Set the week to start on Monday, 7 March 2022
WEEK_START = date(2022, 3, 7)

# Build the list of seven consecutive dates
DAYS = [WEEK_START + timedelta(days=i) for i in range(7)]


# ————— CONFIG —————
HETERO_GRAPH_PATH = "hetero_graph.gpickle"  # your saved graph from build_graph()

print("Using DAYS:", DAYS)


Using DAYS: [datetime.date(2022, 3, 7), datetime.date(2022, 3, 8), datetime.date(2022, 3, 9), datetime.date(2022, 3, 10), datetime.date(2022, 3, 11), datetime.date(2022, 3, 12), datetime.date(2022, 3, 13)]


In [15]:
from collections import Counter
import pickle
import networkx as nx

# Load your full hetero‐graph
with open(HETERO_GRAPH_PATH, "rb") as f:
    G = pickle.load(f)

# Count edges by sorted node_type tuple
edge_type_counts = Counter()
for u, v, data in G.edges(data=True):
    u_type = G.nodes[u].get("node_type", "UNK")
    v_type = G.nodes[v].get("node_type", "UNK")
    types = tuple(sorted([u_type, v_type]))
    edge_type_counts[types] += 1

# Print results
print("Edge counts by node‐type pair:")
for (t1, t2), cnt in edge_type_counts.items():
    print(f"  {t1:>8}–{t2:<8}: {cnt}")
import pandas as pd




Edge counts by node‐type pair:
    device–room    : 1593
    device–floor   : 1593
    device–property: 1282
     floor–room    : 743


In [16]:
import pickle
import networkx as nx
import pandas as pd

# 1. Load the heterogeneous graph
with open(HETERO_GRAPH_PATH, "rb") as f:
    G = pickle.load(f)
print(f"Loaded graph with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")

# 2. Flatten edges into a DataFrame
edges = [
    (u, v, G.nodes[u]["node_type"], G.nodes[v]["node_type"])
    for u, v in G.edges()
]
edges_df = pd.DataFrame(edges, columns=["u", "v", "type_u", "type_v"])

# 3. Define edge type masks
mask_dev_room   = ((edges_df.type_u=="device") & (edges_df.type_v=="room"))   | \
                  ((edges_df.type_v=="device") & (edges_df.type_u=="room"))
mask_room_floor = ((edges_df.type_u=="room")   & (edges_df.type_v=="floor"))  | \
                  ((edges_df.type_v=="room")   & (edges_df.type_u=="floor"))
mask_dev_prop   = ((edges_df.type_u=="device") & (edges_df.type_v=="property")) | \
                  ((edges_df.type_v=="device") & (edges_df.type_u=="property"))
mask_dev_floor  = ((edges_df.type_u=="device") & (edges_df.type_v=="floor"))    | \
                  ((edges_df.type_v=="device") & (edges_df.type_u=="floor"))

# 4. Extract and annotate subsets
df_dev_room = (
    edges_df[mask_dev_room]
    .rename(columns={"u":"source", "v":"target"})
    .assign(relation="device_room")
)

df_room_floor = (
    edges_df[mask_room_floor]
    .rename(columns={"u":"source", "v":"target"})
    .assign(relation="room_floor")
)

df_dev_prop = (
    edges_df[mask_dev_prop]
    .rename(columns={"u":"source", "v":"target"})
    .assign(relation="device_property")
)

df_dev_floor = (
    edges_df[mask_dev_floor]
    .rename(columns={"u":"source", "v":"target"})
    .assign(relation="device_floor")  # direct device–floor edges
)

# 5. Combine all edges
final_edges = pd.concat([
    df_dev_room,
    df_room_floor,
    df_dev_floor,
    df_dev_prop
], ignore_index=True)

# 6. Log summary
print("Edge counts by relation:\n", final_edges['relation'].value_counts())

# 7. Save to CSV
final_edges.to_csv(EDGE_MASTER, index=False)
print(f"Saved {len(final_edges)} edges (with relations) to {EDGE_MASTER}")


Loaded graph with 2396 nodes and 5211 edges
Edge counts by relation:
 relation
device_room        1593
device_floor       1593
device_property    1282
room_floor          743
Name: count, dtype: int64
Saved 5211 edges (with relations) to edges_master2.csv


In [17]:
def extract_measurement_type(prop_str):
    prop_str = prop_str.lower()
    if "temp" in prop_str:
        return "temperature"
    if "co2" in prop_str:
        return "co2"
    if "humidity" in prop_str:
        return "humidity"
    return "other"

In [18]:

# 2. Regex to split "<scenario_key>__<YYYY-MM-DD>.csv"
pattern = re.compile(r"^(.+)__\d{4}-\d{2}-\d{2}\.csv$")

# 3. Group paths by scenario_key
scenarios = defaultdict(list)
for filepath in all_paths:
    fname = os.path.basename(filepath)
    m = pattern.match(fname)
    if not m:
        continue
    key = m.group(1)
    scenarios[key].append(filepath)

# 4. Sort the file lists (so days are in order) and scenario keys
for key in scenarios:
    scenarios[key] = sorted(scenarios[key])
scenario_keys = sorted(scenarios.keys())

# 5. Summary
print(f"Discovered {len(scenario_keys)} unique scenarios.")
for key in scenario_keys:
    print(f"  • {key}: {len(scenarios[key])} days")

# Now `scenarios` is a dict mapping each scenario_key → list of its 7 daily CSV paths


Discovered 28 unique scenarios.
  • CO2_Humidity_temp__floors1: 7 days
  • CO2_Humidity_temp__floors1_2_3_4_5_6_7: 7 days
  • CO2_Humidity_temp__floors1_3: 7 days
  • CO2_Humidity_temp__floors2: 7 days
  • CO2_Humidity_temp__floors2_3_4: 7 days
  • CO2_Humidity_temp__floors3: 7 days
  • CO2_Humidity_temp__floors5_6: 7 days
  • CO2__floors1: 7 days
  • CO2__floors1_2_3_4_5_6_7: 7 days
  • CO2__floors1_3: 7 days
  • CO2__floors2: 7 days
  • CO2__floors2_3_4: 7 days
  • CO2__floors3: 7 days
  • CO2__floors5_6: 7 days
  • Humidity__floors1: 7 days
  • Humidity__floors1_2_3_4_5_6_7: 7 days
  • Humidity__floors1_3: 7 days
  • Humidity__floors2: 7 days
  • Humidity__floors2_3_4: 7 days
  • Humidity__floors3: 7 days
  • Humidity__floors5_6: 7 days
  • temp__floors1: 7 days
  • temp__floors1_2_3_4_5_6_7: 7 days
  • temp__floors1_3: 7 days
  • temp__floors2: 7 days
  • temp__floors2_3_4: 7 days
  • temp__floors3: 7 days
  • temp__floors5_6: 7 days


In [19]:
# Cell 2 (updated): Build and save 7-day “weekly” feature matrices, and log skipped scenarios

skipped = []

for key in scenario_keys:
    # Collect only the 7 files matching our DAYS
    paths = []
    for p in scenarios[key]:
        day_str = os.path.basename(p).rsplit("__", 1)[-1].replace(".csv", "")
        day = datetime.date.fromisoformat(day_str)
        if day in DAYS:
            paths.append(p)
    if len(paths) < len(DAYS):
        skipped.append((key, len(paths)))
        continue

    # Load and concat daily stats
    dfs = []
    for p in sorted(paths):
        df = pd.read_csv(p, parse_dates=['day'])
        df['day'] = df['day'].dt.date
        dfs.append(df)
    
    feat = pd.concat(dfs, ignore_index=True)
    feat['property'] = feat['property'].apply(extract_measurement_type)


    # Pivot to wide format: index=device, columns=stat_day
    wide = feat.pivot(
    index=['device','property'],
    columns='day',
    values=['mean','std','min','max','count']
    )
    wide.columns = [f"{stat}_{d}" for stat, d in wide.columns]
    wide = wide.fillna(0)

    # Save wide table to CSV for downstream use
    out_path = os.path.join(WEEKLY_DIR, f"{key}__weekly.csv")
    wide.to_csv(out_path)
    print(f"Saved weekly features for {key}: {wide.shape}")

# After loop, report skipped scenarios
if skipped:
    print("\n The following scenarios were skipped (only had <7 days):")
    for key, count in skipped:
        print(f"  • {key}: {count}/{len(DAYS)} days found")
else:
    print("\nAll scenarios processed successfully!")


Saved weekly features for CO2_Humidity_temp__floors1: (155, 35)
Saved weekly features for CO2_Humidity_temp__floors1_2_3_4_5_6_7: (725, 35)
Saved weekly features for CO2_Humidity_temp__floors1_3: (304, 35)
Saved weekly features for CO2_Humidity_temp__floors2: (91, 35)
Saved weekly features for CO2_Humidity_temp__floors2_3_4: (383, 35)
Saved weekly features for CO2_Humidity_temp__floors3: (149, 35)
Saved weekly features for CO2_Humidity_temp__floors5_6: (164, 35)
Saved weekly features for CO2__floors1: (48, 35)
Saved weekly features for CO2__floors1_2_3_4_5_6_7: (222, 35)
Saved weekly features for CO2__floors1_3: (97, 35)
Saved weekly features for CO2__floors2: (29, 35)
Saved weekly features for CO2__floors2_3_4: (124, 35)
Saved weekly features for CO2__floors3: (49, 35)
Saved weekly features for CO2__floors5_6: (45, 35)
Saved weekly features for Humidity__floors1: (48, 35)
Saved weekly features for Humidity__floors1_2_3_4_5_6_7: (222, 35)
Saved weekly features for Humidity__floors1_3: 

In [20]:


df["measurement_type"] = df["property"].apply(extract_measurement_type)

In [21]:
import os, re, pandas as pd, torch
import networkx as nx
from torch_geometric.data import HeteroData


def build_hetero_subgraph(
    scenario_csv: str,
    master_edges: pd.DataFrame,
    G: nx.Graph
) -> HeteroData:
    """
    Full pipeline for per-scenario HeteroData subgraph:
      1) Parse key → n_props, n_floors → decide inclusion flags
      2) Load weekly CSV → sensor features + device set
      3) Count sensors per room → decide include_dev_room
      4) Filter master_edges → raw edge lists for dr, dp, rf
      5) Map node IDs to contiguous local indices
      6) Assemble edge_index tensors
      7) Build and return HeteroData
    """
    # --- 1) Key parsing
    fname = os.path.basename(scenario_csv)
    key = fname.replace("__weekly.csv", "")
    m_f = re.search(r"_floors(.+)$", key)
    n_floors = len(m_f.group(1).split("_")) if m_f else 1
    props_part = key.split("_floors")[0]
    n_props = len(props_part.split("_"))

    include_dev_prop = (n_props >= 2)
    include_room_floor = (n_floors >= 2)

    # --- 2) Load and normalize sensor features
    feat_df = pd.read_csv(scenario_csv, index_col="device")
    def _local(u): return u.split("/")[-1] if isinstance(u,str) else u
    feat_df.index = feat_df.index.map(_local)
    devices = set(feat_df.index)

    # --- 3) Device–room inclusion via counts
    if 'src_local' not in master_edges:
        master_edges['src_local'] = master_edges['source'].map(_local)
        master_edges['tgt_local'] = master_edges['target'].map(_local)
    dev_room = master_edges[
        (((master_edges.u_type=='device') & master_edges.src_local.isin(devices)) |
         ((master_edges.v_type=='device') & master_edges.tgt_local.isin(devices))) &
        (((master_edges.u_type=='room')   & ~master_edges.src_local.isin(devices)) |
         ((master_edges.v_type=='room')   & ~master_edges.tgt_local.isin(devices)))
    ]
    room_counts = dev_room['src_local'].where(master_edges['u_type']=='room', dev_room['tgt_local']).value_counts()
    include_dev_room = (room_counts.max() if not room_counts.empty else 0) > 1

    print(f"[{key}] props={n_props}, floors={n_floors} "
          f"=> include_dp={include_dev_prop}, dr={include_dev_room}, rf={include_room_floor}")

    # --- 4) Collect raw edges
    raw_sr, raw_rs = [], []
    raw_sp, raw_ps = [], []
    raw_rf, raw_fr = [], []
    for _, row in master_edges.iterrows():
        src, tgt = _local(row.source), _local(row.target)
        tu, tv = row.u_type, row.v_type
        # dr
        if include_dev_room and {tu,tv}=={'device','room'}:
            if tu=='device' and src in devices: raw_sr.append((src,tgt)); raw_rs.append((tgt,src))
            elif tv=='device' and tgt in devices: raw_sr.append((tgt,src)); raw_rs.append((src,tgt))
        # dp
        if include_dev_prop and {tu,tv}=={'device','property'}:
            if tu=='device' and src in devices: raw_sp.append((src,tgt)); raw_ps.append((tgt,src))
            elif tv=='device' and tgt in devices: raw_sp.append((tgt,src)); raw_ps.append((src,tgt))
        # rf
        if include_room_floor and {tu,tv}=={'room','floor'}:
            if tu=='room': raw_rf.append((src,tgt)); raw_fr.append((tgt,src))
            else:          raw_rf.append((tgt,src)); raw_fr.append((src,tgt))

    # --- 5) Index mapping
    dev2idx  = {d:i for i,d in enumerate(sorted(devices))}
    room_nodes = {r for (_,r) in raw_sr}
    prop_nodes = {p for (_,p) in raw_sp}
    floor_nodes= {f for (_,f) in raw_rf}
    room2idx = {r:i for i,r in enumerate(sorted(room_nodes))} if include_dev_room   else {}
    prop2idx = {p:i for i,p in enumerate(sorted(prop_nodes))} if include_dev_prop   else {}
    floor2idx= {f:i for i,f in enumerate(sorted(floor_nodes))}if include_room_floor else {}

    # --- 6) Build edge_index tensors
    H = HeteroData()
    H['sensor'].x = torch.tensor(feat_df.values, dtype=torch.float)
    def _add_edges(src, rel, dst, raw, idx_map_src, idx_map_dst):
        pairs = [(idx_map_src[s], idx_map_dst[t]) for s,t in raw]
        if pairs: H[src, rel, dst].edge_index = torch.tensor(pairs, dtype=torch.long).t().contiguous()
    if include_dev_room:   _add_edges('sensor','to','room',     raw_sr+raw_rs, dev2idx, room2idx)
    if include_dev_prop:   _add_edges('sensor','to','property', raw_sp+raw_ps, dev2idx, prop2idx)
    if include_room_floor: _add_edges('room','to','floor',     raw_rf+raw_fr,  room2idx, floor2idx)

    return H


def project_sensor_graph(
    hetero: HeteroData,
    include_floor_edges: bool = False,
    k_nn: int = 3
) -> Data:
    """
    Project HeteroData into a homogeneous sensor–sensor graph:
      • Connect sensors sharing a room (if sensor→room exists)
      • Optionally connect sensors sharing a floor
      • Fallback to k-NN for isolated sensors
    """
    X = hetero['sensor'].x
    N = X.size(0)
    print(f"> Projecting sensor graph: N_sensors={N}, include_floor_edges={include_floor_edges}")

    # 1) Room-based groups
    room2sens = defaultdict(list)
    if ('sensor','to','room') in hetero.edge_types:
        for s,r in hetero['sensor','to','room'].edge_index.t().tolist():
            room2sens[r].append(s)
    else:
        print("  ⚠️ no sensor→room edges; skipping room-based links")

    # 2) Floor-based groups
    floor2sens = defaultdict(list)
    if include_floor_edges and ('room','to','floor') in hetero.edge_types:
        for r,f in hetero['room','to','floor'].edge_index.t().tolist():
            for s in room2sens.get(r, []):
                floor2sens[f].append(s)
    elif include_floor_edges:
        print("  ⚠️ include_floor_edges but no room→floor edges")

    # 3) Build undirected pairs
    pairs = set()
    def link_all(groups):
        for members in groups:
            uniq = sorted(set(members))
            for i in range(len(uniq)):
                for j in range(i+1, len(uniq)):
                    pairs.add((uniq[i], uniq[j]))
    link_all(room2sens.values())
    if include_floor_edges:
        link_all(floor2sens.values())
    print(f"  → gathered {len(pairs)} undirected pairs")

    # 4) Fallback for isolated
    connected = {i for (i,j) in pairs} | {j for (i,j) in pairs}
    isolated = set(range(N)) - connected
    if isolated:
        print(f"  ⚠️ {len(isolated)} isolated sensors; applying k-NN fallback (k={k_nn})")
        dists = torch.cdist(X,X)
        for i in isolated:
            dists[i,i] = float('inf')
            for j in dists[i].topk(k_nn, largest=False).indices.tolist():
                pairs.add((i,j)); pairs.add((j,i))

    # 5) Assemble edge_index
    edge_list = [(i,j) for (i,j) in pairs] + [(j,i) for (i,j) in pairs]
    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    print(f"  → final edge count: {edge_index.size(1)}")

    return Data(x=X, edge_index=edge_index)


In [22]:
print(f"\nProcessed {total} scenarios\n")
print("Relation presence counts:")
for rel, cnt in relation_counts.items():
    print(f"  {rel:12s}: {cnt}")
print(f"  scenarios with no hetero‐edges: {len(no_edge_scenarios)}")

print("\nExact edge‐type combinations:")
for combo, cnt in combo_counts.most_common():
    print(f"  {combo}: {cnt}")



Processed 0 scenarios

Relation presence counts:
  scenarios with no hetero‐edges: 0

Exact edge‐type combinations:
